In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [6]:
import numpy as np
import pandas as pd

from src.data.loader import load_data
from src.data.splitter import stratified_train_test_split

from src.preprocessing.cleaning import remove_duplicates
from src.preprocessing.encoding import LabelEncoder
from src.preprocessing.scaling import StandardScaler

ModuleNotFoundError: No module named 'src.preprocessing.encoding'

In [ ]:
df_raw = load_data()
print(f"Kích thước dữ liệu gốc: {df_raw.shape}")
df_raw.head()

In [ ]:
df_clean = remove_duplicates(df_raw)

num_duplicates = len(df_raw) - len(df_clean)
print(f"Số dòng trùng lặp đã loại bỏ: {num_duplicates}")
print(f"Kích thước dữ liệu sau khi làm sạch: {df_clean.shape}")

---
## Bước 3: Encode (Mã hóa nhãn mục tiêu Class)

In [ ]:
feature_names = df_clean.drop(columns=["Class"]).columns.tolist()

# Chuyển X sang NumPy array để tương thích với module splitter
X = df_clean.drop(columns=["Class"]).to_numpy()
y_raw = df_clean["Class"]

# Mã hóa nhãn chuỗi thành số nguyên
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Ánh xạ nhãn (Class Mapping):")
for name, idx in label_mapping.items():
    print(f"  {idx}: {name}")

---
## Bước 4: Split (Phân chia Train / Test phân tầng)

In [ ]:
X_train, X_test, y_train, y_test = stratified_train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Tập huấn luyện (Train): X = {X_train.shape}, y = {y_train.shape}")
print(f"Tập kiểm thử   (Test) : X = {X_test.shape},  y = {y_test.shape}")

In [ ]:
# Kiểm tra xác nhận phân tầng tỉ lệ các giống đậu giữa Train và Test
print(f"{'Class':<12} {'Train Count':<14} {'Train %':<10} {'Test Count':<14} {'Test %':<10}")
print("-" * 60)
for cls_idx, cls_name in enumerate(label_encoder.classes_):
    n_tr = np.sum(y_train == cls_idx)
    n_te = np.sum(y_test == cls_idx)
    pct_tr = n_tr / len(y_train) * 100
    pct_te = n_te / len(y_test) * 100
    print(f"{cls_name:<12} {n_tr:<14} {pct_tr:.2f}%     {n_te:<14} {pct_te:.2f}%")

---
## Bước 5: Scale (Chuẩn hóa đặc trưng - StandardScaler)

In [ ]:
scaler = StandardScaler()

# Fit và transform trên tập Train
X_train_scaled = scaler.fit_transform(X_train)

# CHỈ transform trên tập Test (tránh Data Leakage)
X_test_scaled = scaler.transform(X_test)

print("Kiểm tra phân phối sau chuẩn hóa:")
print(f"Mean tập Train (kỳ vọng ~ 0): {np.allclose(X_train_scaled.mean(axis=0), 0, atol=1e-7)}")
print(f"Std tập Train  (kỳ vọng = 1): {np.allclose(X_train_scaled.std(axis=0), 1, atol=1e-7)}")

---
## Bước 6: Save (Lưu trữ dữ liệu vào thư mục data/processed/)

In [ ]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "dry_bean_processed.npz"
np.savez_compressed(
    output_file,
    X_train=X_train_scaled,
    X_test=X_test_scaled,
    y_train=y_train,
    y_test=y_test,
    features=np.array(feature_names),
    classes=label_encoder.classes_
)

print(f"✅ Đã lưu dữ liệu tiền xử lý thành công vào: {output_file.resolve()}")

In [ ]:
# Kiểm tra nạp lại dữ liệu từ file vừa lưu để sẵn sàng đưa vào mô hình
data = np.load(output_file)
print("Các mảng dữ liệu có sẵn để train model:", data.files)
print("X_train shape:", data["X_train"].shape)
print("X_test shape: ", data["X_test"].shape)
print("y_train shape:", data["y_train"].shape)
print("y_test shape: ", data["y_test"].shape)